# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema accessible at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', '(N/A)')}")
print(f"Date published: {getattr(metadata, 'datePublished', '(N/A)')}")
print("\nKeywords:")
pprint.pprint(getattr(metadata, 'keywords', ''))


## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

We'll inspect the defined record sets and the fields each one provides. Refer to each entity by its `@id` as defined in the Croissant metadata.

In [ ]:
# Gather and inspect record sets, fields, and columns by `@id`
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print('No record sets were found in the Croissant metadata. Attempting to read from distribution files as a fallback.')
    # Sometimes, the files/record structure can be deduced from distribution
    dists = getattr(metadata, 'distribution', [])
    for dist in dists:
        dist_id = getattr(dist, '@id', None)
        print(f"Distribution @id: {dist_id}")
    print("Please see the dataset documentation for details.")
else:
    for recset in record_sets:
        recset_id = getattr(recset, '@id', str(recset))
        print(f"\nRecordSet @id: {recset_id}")
        recset_name = getattr(recset, 'name', '(N/A)')
        print(f"  Name: {recset_name}")
        recset_desc = getattr(recset, 'description', '(N/A)')
        print(f"  Description: {recset_desc}")
        # List fields
        fields = getattr(recset, 'field', [])
        for f in fields:
            field_id = getattr(f, '@id', str(f))
            print(f"    Field @id: {field_id}, Name: {getattr(f, 'name', '(N/A)')}")
        # List columns, if defined
        columns = getattr(recset, 'column', [])
        for c in columns:
            column_id = getattr(c, '@id', str(c))
            print(f"    Column @id: {column_id}, Name: {getattr(c, 'name', '(N/A)')}")


## 3. Data Extraction
Extract data from the available record sets into DataFrames for analysis.

We'll use a list of record set `@id`s referenced above (if present). We'll look for distributions/files if direct record sets aren't found, as is common in some Croissant datasets.

In [ ]:
# First, get list of available record set @ids
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No formal record sets in metadata. Attempting fallback using main distribution file...")
    dists = getattr(metadata, 'distribution', [])
    # Use the first distribution as the main table
    main_rs_id = dists[0]['@id'] if dists else None
    print(f"Using main distribution @id: {main_rs_id}")
    record_set_ids = [main_rs_id]
else:
    # If the record sets are objects (not just @id references), extract @id; else use directly
    record_set_ids = [getattr(rs, '@id', rs) for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    # The following will throw if record_set_id is invalid or not downloadable
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
            display_cols = dataframes[record_set_id].columns[:8]  # show top 8 columns
            display(dataframes[record_set_id][display_cols].head())
        else:
            print("No records found.")
    except Exception as e:
        print(f"  Could not extract {record_set_id}: {e}")

# Use the first loaded dataframe for subsequent analysis
if dataframes:
    selected_record_set = list(dataframes.keys())[0]
else:
    selected_record_set = None

## 4. Exploratory Data Analysis (EDA)

Let's filter, normalize, and group the data on key numeric and categorical fields. All fields are referenced by their `@id` or exact distributed column name.

Typical numeric fields in proteomics datasets are molecular weight (`MW`), coverage percentage, or peptide counts. We'll attempt to use 'MW [kDa]' or a similar column as an example.

In [ ]:
import numpy as np

if selected_record_set is not None and selected_record_set in dataframes:
    df = dataframes[selected_record_set]

    # Try to detect a numeric field for demonstration, prioritizing MW or coverage
    numeric_cols = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    # Fallback to common column names if dtypes are object
    if not numeric_cols:
        for candidate in ['MW [kDa]', 'Coverage [%]', 'Number of Peptides', 'MW', 'Coverage', 'count']:
            if candidate in df.columns:
                numeric_field = candidate
                break
        else:
            print('No clear numeric field found.')
            numeric_field = None
    else:
        numeric_field = numeric_cols[0]

    # Choose a group field (e.g., sample condition, protein category) if available
    for candidate in ['Sample', 'Condition', 'Protein Class', 'Gene Names', 'Accession', 'accession', 'description']:
        if candidate in df.columns:
            group_field = candidate
            break
    else:
        group_field = None

    # Analysis: Filter records for high MW [kDa] values (if MW present)
    threshold = 60
    if numeric_field and numeric_field in df.columns:
        # Convert, force errors='coerce' to handle input as string
        field_data = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[field_data > threshold].copy()
        print(f"Filtered records with '{numeric_field}' > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (field_data[field_data > threshold] - field_data.mean()) / field_data.std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
        
        # Group analysis
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"Data grouped by '{group_field}' (mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print('No group field found in columns for grouping.')
    else:
        print('No numeric field found suitable for analysis.')
else:
    print('Dataframe unavailable for EDA.')

## 5. Visualization

Visualize the distribution of the main numeric field (e.g., MW [kDa]) and how it varies across groupings if a group field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set is not None and selected_record_set in dataframes and numeric_field:
    df = dataframes[selected_record_set]
    plot_series = pd.to_numeric(df[numeric_field], errors='coerce').dropna()

    plt.figure(figsize=(7,4))
    sns.histplot(plot_series, bins=40, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # Boxplot by group, if group_field set
    if group_field and group_field in df.columns:
        if df[group_field].nunique() < 30:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=df[group_field], y=plot_series)
            plt.xticks(rotation=40)
            plt.title(f"{numeric_field} by {group_field}")
            plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR<sup>2</sup> dataset on protein mass spectrometry from human mast cell extracellular vesicles. Using the Croissant schema and `mlcroissant`, we identified and extracted available record sets, and performed basic statistical analysis and visualization on protein molecular weight or coverage fields.

For in-depth analysis, reference fields and columns by their Croissant `@id` or name as shown above. For additional tasks (e.g., merging with external metadata, protein function annotation), refer to dataset documentation or reach out to the authors cited in the Croissant metadata.
